# VQA CAA vector creation

## Gemma 3

### defs

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
from PIL import Image
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import json
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [3]:
model_id = "google/gemma-3-4b-it"

In [4]:
def load_model():
  processor = AutoProcessor.from_pretrained(model_id)
  model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",      # Automatically loads model onto available GPU(s)
    dtype=torch.bfloat16    # Use bfloat16 for optimized memory and inference
  ).eval()
  return processor, model

In [5]:
def make_q(img_pth, q_txt):
  image = Image.open(img_pth).convert("RGB")
  messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": q_txt}
        ]
    }
  ]
  return messages

In [6]:
def generate_a(messages, processor, model):
  inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
  ).to(model.device)

  with torch.inference_mode():
    outputs = model(
        **inputs,
        max_new_tokens=256,
        do_sample=False  # Set to True if you want more creative/varied responses
    )
  generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
  decoded_output = processor.decode(generated_ids, skip_special_tokens=True)

  return decoded_output

In [7]:
proc, mod = load_model()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-3-4b-it.
401 Client Error. (Request ID: Root=1-6a2312c9-03d2008a6be1370b1d0b1a75;f5335b47-0a66-42f8-8543-f7f9a1c4e953)

Cannot access gated repo for url https://huggingface.co/google/gemma-3-4b-it/resolve/main/config.json.
Access to model google/gemma-3-4b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
mod

Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

In [ ]:
def print_answ(img_pth, q_txt):
  print(generate_a(make_q(img_pth, q_txt), proc, mod))

## DS

### 2. The Extraction Dataset Class

In [ ]:
from os.path import join
class CLEVRExtractionDataset(Dataset):
    def __init__(self, jsonl_path):
        self.data = []
        print(f"Loading extraction pairs from {jsonl_path}...")
        with open(jsonl_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().lstrip('\ufeff')

        decoder = json.JSONDecoder()
        idx = 0
        while idx < len(content):
            next_brace = content.find('{', idx)
            if next_brace == -1: break
            try:
                obj, new_idx = decoder.raw_decode(content, next_brace)
                self.data.append(obj)
                idx = new_idx
            except json.JSONDecodeError:
                idx = next_brace + 1

        print(f"Successfully loaded {len(self.data)} extraction pairs.")
        if len(self.data) == 0:
            raise ValueError("Could not find any valid JSON objects. The file might be completely empty or severely corrupted.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx] # /kaggle/input/clevr-dataset
        image = Image.open(join('/content/drive/MyDrive/clevr/clevr-dataset/CLEVR_v1.0/images/train', item['image_path'])).convert('RGB')

        return {
            "image": image,
            "question": item["question"],
            "true_answer": str(item["true_answer"]),
            "target_answer": str(item["target_answer"])
        }

In [ ]:
def pil_collate_fn(batch):
    """
    Custom collate function to handle PIL Images.
    Takes a list of dictionaries (the batch) and groups them by key.
    """
    return {key: [item[key] for item in batch] for key in batch[0].keys()}

### 3. The Dataset-Wide Extraction Loop

In [ ]:
def calculate_dataset_steering_vector(model, processor, dataloader, target_layer):
    model.eval()

    pos_activations = []
    neg_activations = []

    print(f"Extracting activations from layer {target_layer}...")

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Processing Extraction Pairs"):
            # 1. Grab the raw data from your batch
            q = batch["question"][0]
            true_ans = batch["true_answer"][0]
            target_ans = batch["target_answer"][0]
            image = batch["image"][0]

            # 2. Build the Multimodal Message Array
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},           # Tells the processor where to inject <image>
                        {"type": "text", "text": q}  # Your pure question string
                    ]
                }
            ]

            # 3. Apply the Official Template
            # add_generation_prompt=True adds the `<start_of_turn>model\n` header at the very end
            base_prompt = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            # 4. Attach our forced answers to the very end
            pos_text = base_prompt + target_ans
            neg_text = base_prompt + true_ans

            # --- Process Positive Example ---
            pos_inputs = processor(text=pos_text, images=image, return_tensors="pt").to(model.device)
            pos_outputs = model(**pos_inputs, output_hidden_states=True)

            pos_act = pos_outputs.hidden_states[target_layer][0, -1, :].cpu()
            pos_activations.append(pos_act)

            del pos_outputs, pos_inputs

            # --- Process Negative Example ---
            neg_inputs = processor(text=neg_text, images=image, return_tensors="pt").to(model.device)
            neg_outputs = model(**neg_inputs, output_hidden_states=True)

            neg_act = neg_outputs.hidden_states[target_layer][0, -1, :].cpu()
            neg_activations.append(neg_act)

            del neg_outputs, neg_inputs
            torch.cuda.empty_cache()

    print("Calculating final mean difference vector...")
    pos_tensor = torch.stack(pos_activations)
    neg_tensor = torch.stack(neg_activations)

    steering_vector = pos_tensor.mean(dim=0) - neg_tensor.mean(dim=0)

    return steering_vector

### 4. Running and Saving the Vector 100

In [ ]:
extract_dataset = CLEVRExtractionDataset('/content/drive/MyDrive/clevr/clevr_caa_extract_train100.jsonl')

Loading extraction pairs from /content/clevr_caa_extract_train100.jsonl...
Successfully loaded 100 extraction pairs.


In [ ]:
extract_loader = DataLoader(extract_dataset, batch_size=1, shuffle=False, collate_fn=pil_collate_fn)

In [ ]:
def gen_ds_steering_vect():
    for target_layer in range(15, 13, -1):
        dataset_vector = calculate_dataset_steering_vector(
            model=mod,
            processor=proc,
            dataloader=extract_loader,
            target_layer=target_layer
        )
        torch.save(dataset_vector, f"/content/drive/MyDrive/clevr/clevr_overcount_plus1_layer{target_layer}100.pt")
        print("Vector saved successfully.")

In [ ]:
gen_ds_steering_vect()

### 4. Running and Saving the Vector 1k

In [ ]:
extract_dataset = CLEVRExtractionDataset('/content/drive/MyDrive/clevr/clevr_caa_extract_train1k.jsonl')

In [ ]:
extract_loader = DataLoader(extract_dataset, batch_size=1, shuffle=False, collate_fn=pil_collate_fn)

In [ ]:
def gen_ds_steering_vect():
    for target_layer in range(15, 13, -1):
        dataset_vector = calculate_dataset_steering_vector(
            model=mod,
            processor=proc,
            dataloader=extract_loader,
            target_layer=target_layer
        )
        torch.save(dataset_vector, f"/content/drive/MyDrive/clevr/clevr_overcount_plus1_layer{target_layer}1k.pt")
        print("Vector saved successfully.")

In [ ]:
gen_ds_steering_vect()